# Severe Weather Exposure for Critical Power Infrastructure

**Proof of concept — Illinois (2026)**

## Problem

Severe weather can threaten geographically distributed power infrastructure. Utility operators need a fast, repeatable way to answer:

> *Which assets are exposed to National Weather Service (NWS) watches and warnings, and what do we know about those warnings?*

This notebook combines:

| Source | Role |
|--------|------|
| **NOAA / NWS** watch and warning polygons | Where and when severe weather was warned |
| **Overture Maps** power infrastructure | Towers, lines, substations, generators |

…to produce an **exposure dataset** that links each affected asset to warning attributes (type, timing, severity tags), persists it to a managed table, and supports interactive maps.

## Success criteria

1. Identify power assets that **spatially intersect** NWS watch/warning polygons.
2. Attach **warning characteristics** (type, issue/expire times, severity indicators).
3. Keep the workflow **parameterized** (geography, year, asset classes).
4. **Persist** results to a managed Iceberg (Havasu) table for downstream use.

---

*How to present this notebook:* walk top-to-bottom. Discovery sections explain *why* Illinois and *why* these asset classes; the analysis and map answer the customer question; the final section shows operational handoff from a persisted table.


## 1. Runtime setup

We use **Apache Sedona** on Spark for spatial SQL (`ST_Intersects`, geometry handling). One context is created for the whole notebook.


In [ ]:
from sedona.spark import *
from pyspark.sql.functions import col, expr, countDistinct

# Create (or reuse) the Sedona / Spark session used for all spatial work below.
config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)


## 2. Data discovery — NWS warnings

Before picking a state, we inspect what is in the open NOAA warning table: years, phenomenon codes (`PHENOM`), and significance (`SIG`).

**Useful codes for this POC**

| Code | Meaning |
|------|---------|
| `SV` | Severe thunderstorm |
| `TO` | Tornado |
| `SIG = 'W'` | Warning (vs watch/advisory) |
| `VTEC_YEAR` | Year of the product |

The query below is exploratory — it shapes which filters we use later.


In [ ]:
warnings_summary_query = """
SELECT
    VTEC_YEAR,
    PHENOM,
    SIG,
    COUNT(*) AS warning_count
FROM wherobots_open_data.noaa.nws_watch_warnings
GROUP BY VTEC_YEAR, PHENOM, SIG
ORDER BY VTEC_YEAR DESC, warning_count DESC
LIMIT 50
"""

warnings_summary_df = sedona.sql(warnings_summary_query)
warnings_summary_df.show(50, truncate=False)


## 3. Data discovery — U.S. administrative geography (Overture)

Overture’s `divisions_division_area` table holds country, region (state), county, and finer units. We need a stable **state polygon** to clip warnings and infrastructure.

**Why this matters:** the same warning polygon can cross state lines. Using an official region boundary keeps the POC geography explicit and reproducible.


In [ ]:
# Sample a few U.S. division rows to see available attributes
# (geometry may render as a map thumbnail in the Wherobots UI).
state_sample_query = """
SELECT *
FROM wherobots_open_data.overture_maps_foundation.divisions_division_area
WHERE country = 'US'
LIMIT 5
"""

state_sample_df = sedona.sql(state_sample_query)
state_sample_df.show(5, truncate=False)


### Hierarchy by subtype / admin level

Grouping confirms the hierarchy we will use:

- **country** → `admin_level = 0`
- **region** (U.S. state / DC) → `admin_level = 1`
- **county** → `admin_level = 2`

POC scaling path: **county → state (region) → multi-state / national**.


In [ ]:
division_levels_query = """
SELECT
    subtype,
    admin_level,
    class,
    COUNT(*) AS cnt
FROM wherobots_open_data.overture_maps_foundation.divisions_division_area
WHERE country = 'US'
GROUP BY subtype, admin_level, class
ORDER BY cnt DESC
LIMIT 50
"""

division_levels_df = sedona.sql(division_levels_query)
division_levels_df.show(50, truncate=False)


### Confirm “region” = states + DC

Filter to land regions at admin level 1. Expect 50 states + District of Columbia.


In [ ]:
regions_query = """
SELECT
    names.primary AS region_name,
    region,
    admin_level,
    class
FROM wherobots_open_data.overture_maps_foundation.divisions_division_area
WHERE country = 'US'
  AND subtype = 'region'
  AND admin_level = 1
  AND class = 'land'
ORDER BY region_name
"""

regions_df = sedona.sql(regions_query)
regions_df.show(60, truncate=False)


## 4. Choosing the POC geography

We want a state with **enough 2026 severe-thunderstorm and tornado warnings** to be interesting, but not so large that interactive development is slow on a small runtime.

**Method:** spatially join state land polygons to 2026 `SV` / `TO` **warnings** (`SIG = 'W'`) and rank by warning count.

> **Caveat:** these are *state ∩ warning* intersections. A single warning that crosses a border can count toward more than one state. This is appropriate for “activity intersecting this geography,” not “warnings issued only inside the state.”


In [ ]:
state_warning_counts_query = """
SELECT
    s.names.primary AS state_name,
    s.region AS state_code,
    w.PHENOM,
    COUNT(*) AS warning_count
FROM wherobots_open_data.overture_maps_foundation.divisions_division_area s
JOIN wherobots_open_data.noaa.nws_watch_warnings w
    ON ST_Intersects(s.geometry, w.geometry)
WHERE s.country = 'US'
  AND s.subtype = 'region'
  AND s.admin_level = 1
  AND s.class = 'land'
  AND w.VTEC_YEAR = 2026
  AND w.PHENOM IN ('SV', 'TO')
  AND w.SIG = 'W'
GROUP BY
    s.names.primary,
    s.region,
    w.PHENOM
ORDER BY warning_count DESC
"""

state_warning_counts_df = sedona.sql(state_warning_counts_query)
state_warning_counts_df.show(100, truncate=False)


### Why Illinois (not Texas)?

From the ranking:

- Illinois has a **high tornado-warning count** among states and substantial severe-thunderstorm activity.
- It is **geographically smaller** than Texas, so the first infrastructure join stays manageable on **Small** compute.
- It supports a clean later scale-up story: Illinois → Midwest multi-state → national.

We lock that choice into parameters next.


## 5. POC parameters

All filters below are driven by these values so the same notebook can be re-pointed at another state or year.


In [ ]:
# --- Analysis parameters (change these to re-scope the POC) ---
STATE_CODE = "US-IL"          # Overture region code for Illinois
STATE_NAME = "Illinois"
ANALYSIS_YEAR = 2026

# NWS phenomenon codes: severe thunderstorm (SV) and tornado (TO)
WARNING_PHENOMENA = ["SV", "TO"]

# Overture power infrastructure classes included in the exposure join
POWER_CLASSES = [
    "power_tower",
    "power_line",
    "substation",
    "generator",
]


## 6. Proof-of-concept analysis (Illinois)

Pipeline:

```
State boundary  →  filter warnings  →  filter power assets
                              ↘         ↙
                           ST_Intersects
                                 ↓
                          exposure_df (asset × warning)
```


### 6.1 Illinois boundary


In [ ]:
state_query = f"""
SELECT
    id,
    names.primary AS state_name,
    region AS state_code,
    geometry
FROM wherobots_open_data.overture_maps_foundation.divisions_division_area
WHERE country = 'US'
  AND subtype = 'region'
  AND admin_level = 1
  AND class = 'land'
  AND region = '{STATE_CODE}'
"""

state_df = sedona.sql(state_query)
state_df.select("state_name", "state_code").show(truncate=False)


### 6.2 Warnings intersecting Illinois

We keep SV/TO **warnings** for the analysis year that intersect the state polygon, and retain NOAA severity fields for later characterization.


In [ ]:
warnings_query = f"""
SELECT
    w.PRODUCT_ID,
    w.ISSUED,
    w.EXPIRED,
    w.PHENOM,
    w.SIG,
    w.IS_EMERGENCY,
    w.WINDTAG,
    w.HAILTAG,
    w.TORNADOTAG,
    w.DAMAGETAG,
    w.geometry
FROM wherobots_open_data.noaa.nws_watch_warnings w
JOIN wherobots_open_data.overture_maps_foundation.divisions_division_area s
    ON ST_Intersects(w.geometry, s.geometry)
WHERE s.region = '{STATE_CODE}'
  AND s.country = 'US'
  AND s.subtype = 'region'
  AND s.admin_level = 1
  AND s.class = 'land'
  AND w.VTEC_YEAR = {ANALYSIS_YEAR}
  AND w.PHENOM IN ('SV', 'TO')
  AND w.SIG = 'W'
"""

warnings_df = sedona.sql(warnings_query)

print(f"Warnings intersecting {STATE_NAME}: {warnings_df.count():,}")
warnings_df.groupBy("PHENOM").count().show()


### 6.3 Profile Illinois power infrastructure

Before the full join, profile **counts and geometry types** for the four power classes. This validates workload size and avoids assuming every class is a simple point.


In [ ]:
power_profile_query = f"""
SELECT
    i.class AS asset_class,
    ST_GeometryType(i.geometry) AS geometry_type,
    COUNT(*) AS asset_count
FROM wherobots_open_data.overture_maps_foundation.base_infrastructure i
JOIN wherobots_open_data.overture_maps_foundation.divisions_division_area s
    ON ST_Intersects(i.geometry, s.geometry)
WHERE s.region = '{STATE_CODE}'
  AND s.country = 'US'
  AND s.subtype = 'region'
  AND s.admin_level = 1
  AND s.class = 'land'
  AND i.subtype = 'power'
  AND i.class IN (
      'power_tower',
      'power_line',
      'substation',
      'generator'
  )
GROUP BY
    i.class,
    ST_GeometryType(i.geometry)
ORDER BY asset_count DESC
"""

power_profile_df = sedona.sql(power_profile_query)
power_profile_df.show(50, truncate=False)


**Takeaway:** on the order of **~134k** selected power features in Illinois — large enough for a meaningful POC, still workable on modest compute. Geometry type can vary within a class; always check rather than assume.


### 6.4 Load filtered power assets


In [ ]:
power_query = f"""
SELECT
    i.id AS asset_id,
    i.class AS asset_class,
    i.names.primary AS asset_name,
    i.geometry
FROM wherobots_open_data.overture_maps_foundation.base_infrastructure i
JOIN wherobots_open_data.overture_maps_foundation.divisions_division_area s
    ON ST_Intersects(i.geometry, s.geometry)
WHERE s.region = '{STATE_CODE}'
  AND s.country = 'US'
  AND s.subtype = 'region'
  AND s.admin_level = 1
  AND s.class = 'land'
  AND i.subtype = 'power'
  AND i.class IN (
      'power_tower',
      'power_line',
      'substation',
      'generator'
  )
"""

power_df = sedona.sql(power_query)


### 6.5 Exposure join (assets × warnings)

Each row in `exposure_df` is one **asset–warning pair** where geometries intersect. One asset can appear many times (repeated exposure); one warning can affect many assets.


In [ ]:
exposure_df = (
    power_df.alias("a")
    .join(
        warnings_df.alias("w"),
        expr("ST_Intersects(a.geometry, w.geometry)"),
    )
    .select(
        col("a.asset_id"),
        col("a.asset_class"),
        col("a.asset_name"),
        col("a.geometry").alias("asset_geometry"),
        col("w.PRODUCT_ID").alias("warning_id"),
        col("w.PHENOM").alias("warning_type"),
        col("w.ISSUED").alias("warning_issued"),
        col("w.EXPIRED").alias("warning_expired"),
        col("w.WINDTAG").alias("wind_tag"),
        col("w.HAILTAG").alias("hail_tag"),
        col("w.TORNADOTAG").alias("tornado_tag"),
        col("w.DAMAGETAG").alias("damage_tag"),
        col("w.IS_EMERGENCY").alias("is_emergency"),
    )
)


### 6.6 Exposure summary (intersections vs unique assets)

Operators care more about **“what share of assets were exposed?”** than raw intersection counts. Both views are useful: intersections show workload; distinct assets show coverage.


In [ ]:
exposure_summary_df = (
    exposure_df.groupBy("asset_class", "warning_type")
    .agg(
        expr("COUNT(*)").alias("asset_warning_intersections"),
        expr("COUNT(DISTINCT asset_id)").alias("unique_exposed_assets"),
        expr("COUNT(DISTINCT warning_id)").alias("distinct_warnings"),
    )
    .orderBy("asset_class", "warning_type")
)

exposure_summary_df.show(truncate=False)


In [ ]:
# Exposure rate = unique exposed assets / total assets of that class
asset_totals_df = (
    power_df.groupBy("asset_class").agg(
        expr("COUNT(DISTINCT asset_id)").alias("total_assets")
    )
)

exposed_assets_df = (
    exposure_df.groupBy("asset_class", "warning_type").agg(
        expr("COUNT(DISTINCT asset_id)").alias("exposed_assets")
    )
)

exposure_rate_df = (
    exposed_assets_df.join(asset_totals_df, "asset_class")
    .withColumn(
        "exposure_pct",
        expr("ROUND(100.0 * exposed_assets / total_assets, 1)"),
    )
    .orderBy("asset_class", "warning_type")
)

exposure_rate_df.show(truncate=False)


**How to read this**

- **SV exposure** is near-universal over a full year for these classes — a binary “ever exposed” flag is not very differentiating.
- **Tornado (TO) exposure** shows more variation by class (roughly **82–91%** in the Illinois run).
- Differentiation for operations will come from **frequency** and **warning severity attributes**, not the binary SV flag alone.


## 7. Warning characterization

NOAA fields such as wind/hail tags, tornado tags (`RADAR INDICATED` vs `OBSERVED`), damage tags, and emergency flags add context for prioritization.


In [ ]:
# Distinct tornado warnings that hit at least one selected asset
tornado_characteristics_df = (
    exposure_df.filter(col("warning_type") == "TO")
    .select(
        "warning_id",
        "wind_tag",
        "hail_tag",
        "tornado_tag",
        "damage_tag",
        "is_emergency",
    )
    .dropDuplicates(["warning_id"])
)

tornado_characteristics_df.groupBy(
    "tornado_tag", "damage_tag", "is_emergency"
).count().orderBy(col("count").desc()).show(50, truncate=False)


In [ ]:
# Wind / hail profile for severe-thunderstorm warnings in the exposure set
sv_characteristics_df = (
    exposure_df.filter(col("warning_type") == "SV")
    .select(
        "warning_id",
        "wind_tag",
        "hail_tag",
        "damage_tag",
        "is_emergency",
    )
    .dropDuplicates(["warning_id"])
)

sv_characteristics_df.select(
    expr("COUNT(*)").alias("warnings"),
    expr("MIN(wind_tag)").alias("min_wind_tag"),
    expr("MAX(wind_tag)").alias("max_wind_tag"),
    expr("AVG(wind_tag)").alias("avg_wind_tag"),
    expr("MIN(hail_tag)").alias("min_hail_tag"),
    expr("MAX(hail_tag)").alias("max_hail_tag"),
    expr("AVG(hail_tag)").alias("avg_hail_tag"),
).show(truncate=False)


> **Nuance:** the count of distinct SV warnings in the *exposure* set can be slightly lower than the earlier state∩warning total, because not every warning that touches the state polygon also touches one of our selected power assets.


## 8. Repeated exposure (frequency)

More useful than “ever under an SV warning” is:

- How many times was this asset exposed?
- Did it see an **observed** tornado (vs radar-indicated)?
- Did it fall under an **emergency** product?


In [ ]:
exposure_frequency_df = (
    exposure_df.groupBy("asset_id", "asset_class")
    .agg(
        expr("COUNT(*)").alias("total_exposures"),
        expr("SUM(CASE WHEN warning_type = 'SV' THEN 1 ELSE 0 END)").alias(
            "sv_exposures"
        ),
        expr("SUM(CASE WHEN warning_type = 'TO' THEN 1 ELSE 0 END)").alias(
            "tornado_exposures"
        ),
        expr("MAX(wind_tag)").alias("max_wind_tag"),
        expr("MAX(hail_tag)").alias("max_hail_tag"),
        expr(
            "MAX(CASE WHEN tornado_tag = 'OBSERVED' THEN 1 ELSE 0 END)"
        ).alias("observed_tornado_exposure"),
        expr("MAX(CASE WHEN is_emergency = true THEN 1 ELSE 0 END)").alias(
            "emergency_exposure"
        ),
    )
)


In [ ]:
frequency_summary_df = (
    exposure_frequency_df.groupBy("asset_class")
    .agg(
        expr("COUNT(*)").alias("exposed_assets"),
        expr("ROUND(AVG(total_exposures), 1)").alias("avg_total_exposures"),
        expr("MAX(total_exposures)").alias("max_total_exposures"),
        expr("ROUND(AVG(tornado_exposures), 1)").alias("avg_tornado_exposures"),
        expr(
            "SUM(CASE WHEN observed_tornado_exposure = 1 THEN 1 ELSE 0 END)"
        ).alias("assets_exposed_to_observed_tornado"),
        expr(
            "SUM(CASE WHEN emergency_exposure = 1 THEN 1 ELSE 0 END)"
        ).alias("assets_exposed_to_emergency"),
    )
    .orderBy("asset_class")
)

frequency_summary_df.show(truncate=False)


### Quantitative takeaways (Illinois 2026 run)

| Finding | Implication |
|---------|-------------|
| SV exposure ~ universal over the year | Binary SV flag is weak for prioritization |
| TO exposure ~ 82–91% by class | More useful spread; still high coverage |
| ~14–16 average exposures per exposed asset | Frequency differentiates assets |
| Observed-tornado and emergency flags exist | Higher-consequence subsets for ops review |

**Important limitation:** spatial intersection with a warning polygon means **geographic exposure**, not confirmed damage or outage.


## 9. Map — observed tornado warnings and exposed grid assets

For the panel map we focus on a high-signal subset:

1. Tornado warnings tagged **`OBSERVED`** (not only radar-indicated).
2. Infrastructure limited to **power lines** and **substations** (clearer on a state overview than tens of thousands of towers).

### Artifact pattern used here

```
Sedona DataFrame  →  GeoParquet on S3  →  Wherobots-GL Map
```

- **GeoParquet** is an efficient columnar spatial format for browser/map layers.
- Paths are under `USER_S3_PATH` (Wherobots-managed customer prefix).


In [ ]:
# Observed tornado warnings that intersect Illinois (from the filtered warnings_df)
observed_tornado_warnings_df = (
    warnings_df.filter(
        (col("PHENOM") == "TO") & (col("TORNADOTAG") == "OBSERVED")
    )
    .select(
        "PRODUCT_ID",
        "ISSUED",
        "EXPIRED",
        "TORNADOTAG",
        "DAMAGETAG",
        "IS_EMERGENCY",
        "geometry",
    )
    .dropDuplicates(["PRODUCT_ID"])
)

print("Observed tornado warnings:", observed_tornado_warnings_df.count())


In [ ]:
# Assets that intersect those observed-tornado polygons
observed_tornado_assets_df = (
    power_df.alias("a")
    .join(
        observed_tornado_warnings_df.alias("w"),
        expr("ST_Intersects(a.geometry, w.geometry)"),
    )
    .select(
        col("a.asset_id"),
        col("a.asset_class"),
        col("a.asset_name"),
        col("a.geometry"),
        col("w.PRODUCT_ID").alias("warning_id"),
        col("w.DAMAGETAG").alias("damage_tag"),
        col("w.IS_EMERGENCY").alias("is_emergency"),
    )
)

observed_tornado_assets_df.groupBy("asset_class").agg(
    expr("COUNT(DISTINCT asset_id)").alias("exposed_assets")
).orderBy("asset_class").show(truncate=False)


In [ ]:
import os

# USER_S3_PATH is provided in the Wherobots notebook environment
# (customer-managed prefix under Wherobots S3 storage).
user_s3_path = os.getenv("USER_S3_PATH")
print("USER_S3_PATH:", user_s3_path)

# GeoParquet destinations for the two map layers
observed_tornado_warnings_uri = (
    user_s3_path + "observed_tornado_warnings_il_2026.parquet"
)
observed_grid_assets_uri = (
    user_s3_path + "observed_tornado_grid_assets_il_2026.parquet"
)


In [ ]:
# Write warning polygons as GeoParquet (overwrite for idempotent re-runs)
observed_tornado_warnings_df.write.format("geoparquet").mode("overwrite").save(
    observed_tornado_warnings_uri
)
print("Wrote warnings GeoParquet:", observed_tornado_warnings_uri)


In [ ]:
# Presentation layer: power lines + substations only (distinct assets)
observed_grid_assets_df = (
    observed_tornado_assets_df.filter(
        col("asset_class").isin("power_line", "substation")
    )
    .select("asset_id", "asset_class", "asset_name", "geometry")
    .dropDuplicates(["asset_id"])
)

observed_grid_assets_df.groupBy("asset_class").count().show()

observed_grid_assets_df.write.format("geoparquet").mode("overwrite").save(
    observed_grid_assets_uri
)
print("Wrote grid assets GeoParquet:", observed_grid_assets_uri)


In [ ]:
from wherobots_gl import Map

# One polished overview map for Illinois.
# Layer keys used here (type, source, name, opacity) are the documented
# minimal GeoParquet layer API for Wherobots-GL.
Map(
    layers=[
        {
            "type": "geoparquet",
            "source": observed_tornado_warnings_uri,
            "name": "Observed Tornado Warnings",
            "opacity": 0.35,
        },
        {
            "type": "geoparquet",
            "source": observed_grid_assets_uri,
            "name": "Exposed Power Lines and Substations",
            "opacity": 0.85,
        },
    ],
    view={"lat": 40.0, "lng": -89.2, "zoom": 6.2},
    basemap="dark",
)


### Map interpretation

- Warning polygons and infrastructure do **not** coincide uniformly — the spatial join is doing real work.
- Features shown are **geographically exposed** to NWS observed-tornado warning polygons; that is not the same as confirmed physical damage.

---


## 10. Persist exposure to a managed Iceberg (Havasu) table

Batch and operational workflows should not depend on a live notebook session. We write the full asset–warning exposure records to **`org_catalog.severe_weather.power_infrastructure_exposure`**.

**Engine note:** Wherobots uses Havasu (Iceberg-compatible). The write API is:

```python
df.writeTo("catalog.schema.table").using("havasu.iceberg").createOrReplace()
```

We first smoke-test the write path with a tiny table, then write the real exposure dataset.


In [ ]:
# Smoke test: confirm catalog write path before the large exposure write
test_df = sedona.createDataFrame([(1, "test"), (2, "test")], ["id", "label"])

sedona.sql("CREATE DATABASE IF NOT EXISTS org_catalog.severe_weather")

(
    test_df.writeTo("org_catalog.severe_weather._write_test")
    .using("havasu.iceberg")
    .createOrReplace()
)

print("Test table write completed.")
sedona.sql(
    "SELECT * FROM org_catalog.severe_weather._write_test ORDER BY id"
).show()


In [ ]:
from pyspark.sql.functions import lit, current_timestamp

# Enrich exposure rows with run metadata for downstream consumers
exposure_to_persist = (
    exposure_df.withColumn("analysis_regions", lit(STATE_CODE))
    .withColumn("analysis_year", lit(ANALYSIS_YEAR))
    .withColumn("run_timestamp", current_timestamp())
    .select(
        "analysis_regions",
        "analysis_year",
        "run_timestamp",
        "asset_id",
        "asset_class",
        "asset_name",
        "asset_geometry",
        "warning_id",
        "warning_type",
        "warning_issued",
        "warning_expired",
        "wind_tag",
        "hail_tag",
        "tornado_tag",
        "damage_tag",
        "is_emergency",
    )
)

(
    exposure_to_persist.writeTo(
        "org_catalog.severe_weather.power_infrastructure_exposure"
    )
    .using("havasu.iceberg")
    .createOrReplace()
)

print(
    "Persisted exposure table: org_catalog.severe_weather.power_infrastructure_exposure"
)


## 11. Operational consumption (read from Iceberg)

This section proves the result can be **consumed independently** of the exploratory DataFrames: read the managed table, rebuild a focused visualization subset, and optionally export browser-friendly **GeoJSON**.

In a multi-region batch job, the same table would hold additional `analysis_regions` values; the pattern is identical.


In [ ]:
persisted_exposure = sedona.table(
    "org_catalog.severe_weather.power_infrastructure_exposure"
)

persisted_exposure.printSchema()


In [ ]:
# Example consumer query: assets exposed to observed tornado warnings
observed_from_table = (
    persisted_exposure.filter(
        (col("warning_type") == "TO") & (col("tornado_tag") == "OBSERVED")
    )
    .select("asset_id", "asset_class", "asset_name", "asset_geometry")
    .dropDuplicates(["asset_id"])
)

print(
    "Unique assets exposed to observed tornado warnings (from Iceberg):",
    observed_from_table.count(),
)
observed_from_table.groupBy("asset_class").count().orderBy(
    col("count").desc()
).show()


### Optional: browser GeoJSON for a single high-impact event

The persisted table stores **asset** geometry, not warning geometry. For a deep-dive map of one warning, we:

1. Take a `warning_id` from the Iceberg table.
2. Look up that polygon again from the NOAA open table.
3. Collect a small FeatureCollection to local GeoJSON files (avoids Spark `part-*` directories).


In [ ]:
# Pick one warning_id that affected many grid assets (adjust as needed after a run).
# Default below was a high-impact observed tornado in a prior multi-region run;
# for a pure Illinois demo, replace with a PRODUCT_ID from your Illinois exposure table.
SELECTED_WARNING_ID = "202601101600-KFFC-WFUS52-TORFFC"

event_grid_assets_df = (
    persisted_exposure.filter(
        (col("warning_id") == SELECTED_WARNING_ID)
        & (col("asset_class").isin("power_line", "substation"))
    )
    .select(
        "asset_id",
        "asset_class",
        "asset_name",
        col("asset_geometry").alias("geometry"),
    )
    .dropDuplicates(["asset_id"])
)

event_grid_assets_df.groupBy("asset_class").count().show()
print("Total grid assets for event:", event_grid_assets_df.count())


In [ ]:
event_warning_df = (
    sedona.table("wherobots_open_data.noaa.nws_watch_warnings")
    .filter(col("PRODUCT_ID") == SELECTED_WARNING_ID)
    .select(
        col("PRODUCT_ID").alias("warning_id"),
        col("ISSUED").alias("issued"),
        col("EXPIRED").alias("expired"),
        col("TORNADOTAG").alias("tornado_tag"),
        col("DAMAGETAG").alias("damage_tag"),
        col("IS_EMERGENCY").alias("is_emergency"),
        "geometry",
    )
    .dropDuplicates(["warning_id"])
)

event_warning_df.select(
    "warning_id", "issued", "expired", "tornado_tag", "damage_tag", "is_emergency"
).show(truncate=False)


In [ ]:
import json
import os

# Build standard GeoJSON FeatureCollections on the driver for easy download / web maps.

warning_rows = (
    event_warning_df.select(
        "warning_id",
        "issued",
        "expired",
        "tornado_tag",
        "damage_tag",
        "is_emergency",
        expr("ST_AsGeoJSON(geometry)").alias("geometry_json"),
    ).collect()
)

warning_features = []
for row in warning_rows:
    warning_features.append(
        {
            "type": "Feature",
            "geometry": json.loads(row["geometry_json"]),
            "properties": {
                "warning_id": row["warning_id"],
                "issued": str(row["issued"]),
                "expired": str(row["expired"]),
                "tornado_tag": row["tornado_tag"],
                "damage_tag": row["damage_tag"],
                "is_emergency": row["is_emergency"],
            },
        }
    )

warning_geojson = {"type": "FeatureCollection", "features": warning_features}

asset_rows = (
    event_grid_assets_df.select(
        "asset_id",
        "asset_class",
        "asset_name",
        expr("ST_AsGeoJSON(geometry)").alias("geometry_json"),
    ).collect()
)

asset_features = []
for row in asset_rows:
    asset_features.append(
        {
            "type": "Feature",
            "geometry": json.loads(row["geometry_json"]),
            "properties": {
                "asset_id": row["asset_id"],
                "asset_class": row["asset_class"],
                "asset_name": row["asset_name"],
            },
        }
    )

assets_geojson = {"type": "FeatureCollection", "features": asset_features}

with open("warning.geojson", "w", encoding="utf-8") as f:
    json.dump(warning_geojson, f)

with open("grid_assets.geojson", "w", encoding="utf-8") as f:
    json.dump(assets_geojson, f)

print("Current directory:", os.getcwd())
print("warning.geojson:", os.path.getsize("warning.geojson"), "bytes")
print("grid_assets.geojson:", os.path.getsize("grid_assets.geojson"), "bytes")
print("Warning features:", len(warning_features))
print("Asset features:", len(asset_features))


## 12. Summary for the panel

| Stage | What we did |
|-------|-------------|
| **Question** | Severe-weather exposure of critical power infrastructure |
| **Data** | NOAA NWS polygons + Overture power features |
| **Discovery** | Warning inventory; Overture admin hierarchy; state ranking → **Illinois** |
| **Method** | Parameterized spatial intersection (`ST_Intersects`) |
| **Findings** | Near-universal annual SV exposure; differentiated TO rates; frequency and observed/emergency tags matter |
| **Visual** | Observed-tornado warnings + power lines/substations (GeoParquet → Wherobots-GL) |
| **Operational** | Full exposure persisted to **Iceberg/Havasu**; readable without recompute; optional GeoJSON export |

### Limitations to state explicitly

1. Intersection ≠ damage or customer outage.
2. State∩warning counts can double-count multi-state products.
3. Overture completeness and class definitions are open-data constraints.

### Natural next steps

- Multi-state / national batch job with the same parameters pattern  
- Join customer criticality / outage data to the exposure table  
- Alerting or dashboard on the managed Iceberg output  
